# 03 · Modelling and evaluation

Two models, honestly compared:

- **Baseline** — L2-regularised logistic regression on Tier-1 (champion multi-hot + bans)
- **Stronger** — LightGBM on Tier-2 (archetype aggregates + diffs)

against three reference points: a coin flip, a constant base-rate predictor, and a naive
"sum of champion win-rates" rule.

**Evaluation is calibration-forward.** The interesting question is not "how often is it right"
but "when it says 55%, does blue win 55% of the time?" — so log-loss, Brier and the calibration
curve matter more than accuracy here.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config, features as F, model as M, viz
viz.apply_style()
pd.set_option("display.width", 140)

## The evaluation protocol, and why it is built this way

Three deliberate choices:

1. **Out-of-fold cross-validation, not a single split.** Every match gets a prediction from a
   model that never saw it, and every match contributes to the estimate. With a weak signal, a
   single hold-out split is too noisy to distinguish a real effect from luck.
2. **Nested CV.** Regularisation strength is chosen by an *inner* CV on the training rows of each
   outer fold. If we instead tuned on the whole dataset, the reported score would be
   optimistically biased — the model would have peeked at the rows it is scored on.
3. **The naive baseline is refit inside each fold.** Its champion win-rates are outcome-derived,
   so computing them once on all the data would leak the answer into the baseline.

In [2]:
df = F.load_matches()
labels = F.load_labels()
X1, y = F.tier1_features(df)
X2, _ = F.tier2_features(df, labels)
print(f"{len(df):,} matches | Tier-1 {X1.shape} | Tier-2 {X2.shape}")

8,690 matches | Tier-1 (8690, 519) | Tier-2 (8690, 39)


## Results

The cell below loads the comparison produced by `python src/model.py`. That script runs the full
nested cross-validation (several minutes), so the notebook reads its saved output rather than
recomputing it. Re-run the script whenever the data changes.

In [3]:
table = pd.read_csv(config.DATA_PROCESSED / "model_comparison.csv", index_col=0)
headline = json.loads((config.DATA_PROCESSED / "headline.json").read_text())
table.round(4)

,log_loss,roc_auc,brier,accuracy
logreg Tier-2,0.6903,0.5373,0.2486,0.5339
naive champ-WR,0.6919,0.5236,0.2494,0.5206
logreg Tier-1 (M4),0.6921,0.5162,0.2495,0.5158
LightGBM Tier-2 (M5),0.6924,0.5232,0.2496,0.5231
base rate (const),0.6925,0.5000,0.2497,0.5177
coin flip (0.5),0.6931,0.5000,0.2500,0.5177


### Reading the metrics

- **log-loss** (lower is better) — penalises confident mistakes. A coin flip scores 0.6931,
  so that value is the line to beat. This is the headline metric.
- **ROC-AUC** — probability the model ranks a random blue win above a random blue loss.
  0.5 is chance.
- **Brier** — mean squared error of the probabilities; another calibration-sensitive score.
- **accuracy** — included for familiarity, but it is the *least* informative here: with a
  near-balanced label, always guessing the more common side already scores close to the best model.

In [4]:
best = headline["best_model"]
print(f"best model: {best}")
print(f"  ROC-AUC  {headline['roc_auc']:.4f}   ({(headline['roc_auc'] - 0.5) * 100:+.2f} pp vs chance)")
print(f"  log-loss {headline['log_loss']:.4f}  (coin flip {headline['baseline_log_loss']:.4f}, "
      f"constant {headline['constant_log_loss']:.4f})")
print(f"  accuracy {headline['accuracy']:.2%}  (always-guess-common-side {headline['majority_accuracy']:.2%})")
print(f"\npredicted win probability spans {headline['prob_min']:.1%} to {headline['prob_max']:.1%} "
      f"(sd {headline['prob_std']:.3f})")

best model: logreg  Tier-2
  ROC-AUC  0.5373   (+3.73 pp vs chance)
  log-loss 0.6903  (coin flip 0.6931, constant 0.6925)
  accuracy 53.39%  (always-guess-common-side 51.77%)

predicted win probability spans 35.9% to 62.5% (sd 0.033)


## Two findings worth stating plainly

**Gradient boosting lost — but the archetype features won.** LightGBM finished behind plain
logistic regression: the extra model capacity found no signal to exploit. Among the *linear*
models, the archetype representation (Tier 2 — team shape) edged raw champion identity (Tier 1).
At an earlier, smaller sample the two were indistinguishable; with more games, *what kind of team
you built* — engage, hard CC, frontline, AD/AP balance — carries slightly more signal than *which
exact champions* are on it. Reported as-is, not tuned until a preferred model wins.

**Regularisation mattered more than model choice.** An earlier version of this analysis fixed
`C = 0.1` by hand and the model lost to a coin flip on log-loss: it was overconfident, not wrong
on average. Letting the inner CV choose `C` fixed it. With hundreds of sparse features, the penalty
is not a detail to guess.

In [5]:
# Calibration: are the stated probabilities honest?
from sklearn.calibration import calibration_curve
oof = pd.read_csv(config.DATA_PROCESSED / "oof_predictions.csv")
p = oof[best] if best in oof.columns else oof.iloc[:, 2]
prob_true, prob_pred = calibration_curve(oof["win"], np.clip(p, 1e-6, 1 - 1e-6), n_bins=10)

fig, ax = viz.figure(5.6, 5.4)
ax.plot([0, 1], [0, 1], color=viz.INK_MUTED, lw=1, label="perfect")
ax.plot(prob_pred, prob_true, "o-", color=viz.BLUE_SIDE, lw=2, ms=6,
        mec=viz.SURFACE, mew=1.5, label=best)
ax.set_xlabel("win chance the model predicted")
ax.set_ylabel("how often that actually happened")
ax.legend(loc="upper left")
viz.title_block(ax, "Calibration", "On the line = honest probabilities.")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_22832\828593562.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The points sit close to the diagonal — the model is **well calibrated**. It is not lying about its
confidence. It simply never *has* much confidence: notice how little of the 0–1 axis it occupies.

In [6]:
fig, ax = viz.figure(8, 4.2)
ax.hist(p, bins=32, color=viz.BLUE_SIDE, linewidth=0)
viz.reference_line(ax, 0.5, "coin flip", axis="x")
ax.set_xlim(0, 1)
ax.set_xticks([0, .25, .5, .75, 1]); ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("win chance the draft gave the blue team"); ax.set_ylabel("number of games")
viz.title_block(ax, "The draft almost never picks a winner",
                "A draft that decided games would push these toward 0% and 100%.")
plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_22832\2349857972.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Does the draft matter more at higher elo?

Games are tagged with the rank of the seed player who surfaced them (approximate — see notebook 01).
If drafting is more consequential where execution is more consistent, ROC-AUC should rise with tier.

In [7]:
merged = oof.merge(df[["matchId", "seed_tier"]], on="matchId", how="left")
from sklearn.metrics import roc_auc_score

rows = []
for tier, g in merged.groupby("seed_tier"):
    if len(g) < 200 or g["win"].nunique() < 2:
        continue
    rows.append({"tier": tier, "games": len(g),
                 "roc_auc": roc_auc_score(g["win"], g[best] if best in g else g.iloc[:, 2])})
by_tier = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
print("Differences this small on these sample sizes are within noise --")
print("treat as suggestive at best, not a finding.")
by_tier.round(4)

Differences this small on these sample sizes are within noise --
treat as suggestive at best, not a finding.


,tier,games,roc_auc
2,MASTER,5860,0.5440
1,GRANDMASTER,1918,0.5280
0,CHALLENGER,620,0.5199
3,UNKNOWN,292,0.5019


## Conclusion

Champion select alone gets only a few points above chance. The model is honest and well
calibrated; it just has very little to say, because **the draft is not what decides a
solo-queue game**. The remaining variance is execution: mechanics, decisions, tilt,
and the other nine players.

That low ceiling is the finding, not a failure of the model.